In [5]:
from transformers import AutoModelForMultimodalLM, AutoProcessor, logging, BitsAndBytesConfig,GenerationConfig,TextIteratorStreamer
import json
import torch
from PIL import Image
import gradio as gr
from threading import Thread


# avoid having too much output from the transformers library
logging.set_verbosity_warning()

# fairly small but good model
MODEL_NAME = "Qwen/Qwen3.5-0.8B"

quantization_config = None

## If you need to use quantization, you can specify the quantization configuration here
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,  # set to True to use 4-bit quantization, you should use either 4-bit or 8-bit, not both
#     load_in_8bit=True,  # set to True to use 8-bit quantization, you should use either 4-bit or 8-bit, not both
# )


# We're using the `AutoModelForMultimodalLM` class to enable multimodal generation
# If you are not using a multimodal model, you can use AutoModelForCausalLM instead
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"  # automatically uses right device e.g., GPU if available
)

# We're using the `AutoProcessor` class to enable multimodal generation
# Normally, you can use AutoTokenizer
processor = AutoProcessor.from_pretrained(MODEL_NAME)

temp_config = GenerationConfig(
    do_sample=False,
    max_new_tokens=512
   
)

SYSTEM_PROMPT = (
    "You are a mushroom expert. "
    "Your default behavior is to talk about mushrooms and stir any conversation back to mushrooms. "
    "Be helpful, informative, and stay in character."
    "Use natural language and avoid using JSON format in your responses. "
)



Loading weights: 100%|██████████| 473/473 [00:02<00:00, 208.20it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


In [3]:
def process_image_for_json(image):
    json_prompt_conversation = [
        {"role": "system", "content": "You are a precise computer vision data extraction assistant."},
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {
                    "type": "text",
                    "text": (
                        "Analyze this mushroom image and return strictly a valid JSON object without markdown or code blocks. "
                        "The JSON must have the following keys:\n"
                        "- common_name: string\n"
                        "- genus: string\n"
                        "- confidence: float between 0.0 and 1.0\n"
                        "- visible: list of strings subset of ['cap', 'hymenium', 'stipe']\n"
                        "- color: string\n"
                        "- edible: boolean or null\n"
                    )
                }
            ]
        }
    ]
    
    formatted_prompt = processor.apply_chat_template(json_prompt_conversation, add_generation_prompt=True)
    inputs = processor(text=[formatted_prompt], images=[image], return_tensors="pt").to(model.device)
    
    output_ids = model.generate(**inputs, generation_config=temp_config)
    generated_ids = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)]
    json_str = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    
    print("===== Extracted Mushroom JSON  =====")
    print(json_str)
    print("=================================================")
    return json_str

In [ ]:
MODEL_NAME = "Qwen/Qwen3-2B"

quantization_config = None

## If you need to use quantization, you can specify the quantization configuration here
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,  # set to True to use 4-bit quantization, you should use either 4-bit or 8-bit, not both
#     load_in_8bit=True,  # set to True to use 8-bit quantization, you should use either 4-bit or 8-bit, not both
# )


# We're using the `AutoModelForMultimodalLM` class to enable multimodal generation
# If you are not using a multimodal model, you can use AutoModelForCausalLM instead
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"  # automatically uses right device e.g., GPU if available
)

# We're using the `AutoProcessor` class to enable multimodal generation
# Normally, you can use AutoTokenizer
processor = AutoProcessor.from_pretrained(MODEL_NAME)

In [6]:


def bot_response(message, history):
    user_text = message.get("text", "").strip()
    uploaded_files = message.get("files", [])
    
    current_image = None
    json_info = None
    
    if uploaded_files:
        img_path = uploaded_files[0] if isinstance(uploaded_files[0], str) else uploaded_files[0]["path"]
        current_image = Image.open(img_path)
        json_info = process_image_for_json(current_image)

    
    conversation = [{"role": "system", "content": SYSTEM_PROMPT}]
    
    current_content = []
    if current_image:
        current_content.append({"type": "image"})
        
    if json_info:
       
        if user_text:
            instruction = f"The user asked: '{user_text}'. Answer the user's question using this background metadata about the mushroom image: {json_info}"
        else:
            instruction = f"The user uploaded an image without a question. Provide a concise summary of the mushroom in the image using this metadata: {json_info}"
        current_content.append({"type": "text", "text": instruction})
    else:
        current_content.append({"type": "text", "text": user_text})

    conversation.append({"role": "user", "content": current_content})

    
    text_prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = processor(
        text=[text_prompt], 
        images=[current_image] if current_image else None, 
        return_tensors="pt"
    ).to(model.device)

    streamer = TextIteratorStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True)
    generation_kwargs = dict(**inputs, streamer=streamer, generation_config=temp_config)

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    
    partial_text = ""
    for new_text in streamer:
        partial_text += new_text
        yield partial_text


demo = gr.ChatInterface(bot_response, multimodal=True)
if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


===== Extracted Mushroom JSON  =====
```json
[
  {
    "common_name": "Ganoderma",
    "genus": "Ganoderma",
    "confidence": 0.95,
    "visible": ["cap", "hymenium", "stipe"],
    "color": "yellow",
    "edible": true
  }
]
```
